# Hito 2 Modeling Notebook

This notebook trains and evaluates two targets: `is_top10` and `is_top5` using the locked temporal split.
It also produces calibration plots, error analysis slices, and a what-if scenario comparison.

Dataset: `f1_strategy_race_level.csv` (expected in `hito1/`).
Split: train 2019-2021, calibration 2022, test 2023-2024.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score, f1_score
from sklearn.calibration import calibration_curve

In [ ]:
# Locate dataset
data_path_candidates = [
    Path('hito1') / 'f1_strategy_race_level.csv',
    Path('f1_strategy_race_level.csv'),
]
for p in data_path_candidates:
    if p.exists():
        data_path = p
        break
else:
    raise FileNotFoundError('Dataset not found. Put f1_strategy_race_level.csv in hito1/.')

df = pd.read_csv(data_path)
df.head()

In [ ]:
# Feature engineering
df['grid_fallback'] = df['grid_position']
df.loc[df['grid_fallback'].isna(), 'grid_fallback'] = df['qualifying_position']

def count_compounds(seq):
    if pd.isna(seq) or not isinstance(seq, str) or seq.strip() == '':
        return np.nan
    parts = [p.strip() for p in seq.split('-') if p.strip()]
    if not parts:
        return np.nan
    return len(set(parts))

df['num_compounds'] = df['compound_sequence'].apply(count_compounds)

for col in ['grid_fallback', 'n_stops', 'num_compounds']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[['grid_fallback', 'n_stops', 'num_compounds']].describe()

Leakage note: `n_stops` and `compound_sequence` are post-race observations in the raw data.
We use them only as scenario inputs for what-if comparisons, not as pre-race signals.
Calibration uses the 2022 season only, as required by the locked split.

In [ ]:
# Split by season
train = df[df['season'].between(2019, 2021)]
cal = df[df['season'] == 2022]
test = df[df['season'].between(2023, 2024)]

print('train', train.shape, 'cal', cal.shape, 'test', test.shape)

In [ ]:
BASE_FEATURES = ['grid_fallback', 'constructor_tier']
FULL_FEATURES = ['grid_fallback', 'constructor_tier', 'n_stops', 'num_compounds']
CAT_FEATURES = ['constructor_tier']
RANDOM_STATE = 414

def build_base(features):
    numeric = [f for f in features if f not in CAT_FEATURES]
    categorical = [f for f in features if f in CAT_FEATURES]
    pre = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', numeric),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ],
        remainder='drop'
    )
    base = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        solver='liblinear',
        random_state=RANDOM_STATE
    )
    return Pipeline([('pre', pre), ('model', base)])

def fit_platt(base_model, X_cal, y_cal):
    # Manual Platt scaling on the fixed calibration set
    scores = base_model.decision_function(X_cal).reshape(-1, 1)
    platt = LogisticRegression(max_iter=1000, C=1e6, solver='lbfgs', random_state=RANDOM_STATE)
    platt.fit(scores, y_cal)
    return platt

def predict_proba_calibrated(base_model, platt, X):
    scores = base_model.decision_function(X).reshape(-1, 1)
    return platt.predict_proba(scores)[:, 1]

def eval_metrics(proba, y):
    pred = (proba >= 0.5).astype(int)
    return {
        'brier': float(brier_score_loss(y, proba)),
        'roc_auc': float(roc_auc_score(y, proba)),
        'f1_macro': float(f1_score(y, pred, average='macro')),
    }

In [ ]:
def fit_target_models(target):
    # Baseline
    tr_b = train[BASE_FEATURES + [target]].dropna()
    ca_b = cal[BASE_FEATURES + [target]].dropna()
    te_b = test[BASE_FEATURES + [target]].dropna()
    X_tr_b, y_tr_b = tr_b[BASE_FEATURES], tr_b[target].astype(int)
    X_ca_b, y_ca_b = ca_b[BASE_FEATURES], ca_b[target].astype(int)
    X_te_b, y_te_b = te_b[BASE_FEATURES], te_b[target].astype(int)

    base_model = build_base(BASE_FEATURES)
    base_model.fit(X_tr_b, y_tr_b)
    base_platt = fit_platt(base_model, X_ca_b, y_ca_b)
    base_proba = predict_proba_calibrated(base_model, base_platt, X_te_b)

    # Full (scenario features)
    tr_f = train[FULL_FEATURES + [target]].dropna()
    ca_f = cal[FULL_FEATURES + [target]].dropna()
    te_f = test[FULL_FEATURES + [target]].dropna()
    X_tr_f, y_tr_f = tr_f[FULL_FEATURES], tr_f[target].astype(int)
    X_ca_f, y_ca_f = ca_f[FULL_FEATURES], ca_f[target].astype(int)
    X_te_f, y_te_f = te_f[FULL_FEATURES], te_f[target].astype(int)

    full_model = build_base(FULL_FEATURES)
    full_model.fit(X_tr_f, y_tr_f)
    full_platt = fit_platt(full_model, X_ca_f, y_ca_f)
    full_proba = predict_proba_calibrated(full_model, full_platt, X_te_f)

    metrics = {
        'baseline': eval_metrics(base_proba, y_te_b),
        'full': eval_metrics(full_proba, y_te_f),
        'counts': {
            'train': int(len(y_tr_f)),
            'cal': int(len(y_ca_f)),
            'test': int(len(y_te_f)),
        },
    }

    model_pack = {
        'base_model': base_model,
        'base_platt': base_platt,
        'full_model': full_model,
        'full_platt': full_platt,
        'X_test_base': X_te_b,
        'y_test_base': y_te_b,
        'X_test_full': X_te_f,
        'y_test_full': y_te_f,
    }
    return metrics, model_pack

results = {}
models = {}
for target in ['is_top10', 'is_top5']:
    metrics, model_pack = fit_target_models(target)
    results[target] = metrics
    models[target] = model_pack

results_df = (
    pd.DataFrame(results)
    .T
    .applymap(lambda x: x if isinstance(x, dict) else x)
)
results_df

In [ ]:
# Calibration curves for the full model on each target
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, target in enumerate(['is_top10', 'is_top5']):
    pack = models[target]
    proba = predict_proba_calibrated(pack['full_model'], pack['full_platt'], pack['X_test_full'])
    y = pack['y_test_full']
    frac_pos, mean_pred = calibration_curve(y, proba, n_bins=10, strategy='uniform')
    ax = axes[i]
    ax.plot(mean_pred, frac_pos, marker='o', label='calibration')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='ideal')
    ax.set_title(target)
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Observed frequency')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Error analysis slices (full model)
def slice_table(target, slice_col):
    pack = models[target]
    d = test[FULL_FEATURES + [target, 'strategy_type', 'circuit_type', 'wet_laps']].copy()
    d = d.dropna(subset=FULL_FEATURES + [target])
    X = d[FULL_FEATURES]
    y = d[target].astype(int)
    proba = predict_proba_calibrated(pack['full_model'], pack['full_platt'], X)
    pred = (proba >= 0.5).astype(int)
    d['proba'] = proba
    d['pred'] = pred
    d['weather_slice'] = np.where(d['wet_laps'].fillna(0) > 0, 'wet', 'dry')

    def metrics(group):
        y_true = group[target].astype(int)
        p = group['proba']
        y_hat = group['pred']
        return pd.Series({
            'n': len(group),
            'brier': brier_score_loss(y_true, p),
            'f1_macro': f1_score(y_true, y_hat, average='macro')
        })

    return d.groupby(slice_col).apply(metrics).reset_index()

for target in ['is_top10', 'is_top5']:
    print('
Target:', target)
    print('
Strategy type')
    print(slice_table(target, 'strategy_type').sort_values('brier'))
    print('
Circuit type')
    print(slice_table(target, 'circuit_type').sort_values('brier'))
    print('
Weather (wet_laps > 0)')
    print(slice_table(target, 'weather_slice').sort_values('brier'))

In [ ]:
# What-if scenario: Bahrain 2023, Perez (driver_id = 'perez')
row = df[(df['season'] == 2023) & (df['race_name'] == 'Bahrain Grand Prix') & (df['driver_id'] == 'perez')].iloc[0]
base = {
    'grid_fallback': float(row['grid_fallback']) if not pd.isna(row['grid_fallback']) else float(row['qualifying_position']),
    'constructor_tier': row['constructor_tier'],
}
scenario_a = {**base, 'n_stops': 1.0, 'num_compounds': 2.0}
scenario_b = {**base, 'n_stops': 2.0, 'num_compounds': 3.0}
Xa = pd.DataFrame([scenario_a])
Xb = pd.DataFrame([scenario_b])

for target in ['is_top10', 'is_top5']:
    pack = models[target]
    p_a = predict_proba_calibrated(pack['full_model'], pack['full_platt'], Xa)[0]
    p_b = predict_proba_calibrated(pack['full_model'], pack['full_platt'], Xb)[0]
    print(f'{target} | Scenario A: {p_a:.3f} | Scenario B: {p_b:.3f} | diff: {p_b - p_a:+.3f}')

We use the full model for scenario comparisons because it includes strategy inputs.
For pure predictive performance, compare against the baseline results above and the docent is_top10 reference.